# Model Performance Figures (baselines vs text-only models)

Reads `merged_master.pkl` (keys, target, `log_volume`) and every registered prediction file,
restricts to the **common sample** (tweeted stock-days 2012-2022 on which every model has a
prediction, target de-meaned by date) and draws:

1. yearly average daily rank correlation, return-target and rank-target models side by side;
2. monthly average daily rank correlation, 12-month rolling mean;
3. mean de-meaned next-day return by daily prediction decile, one small panel per model;
4. cumulative top-minus-bottom decile return (equal-weighted, sum of daily de-meaned returns);
5. summary: full-sample rank correlation and decile spread with 95% confidence bands, all models.

Figures go to `Figures/text_models/` (next to `Code/`), plus a CSV of the summary table.
Runs headless with `tools/run_notebook.py`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

MODEL_DATA_DIR = Path(r"D:\StockTwits\Data")
FIG_DIR = Path(r"D:\StockTwits\Figures") / "text_models"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_START, SAMPLE_END = "2012-01-01", "2022-12-31"
TARGET = "f_cumret1"

# Registry: key -> (file, label, entity, target). 'entity' fixes the colour across panels;
# 'target' is what the model was trained on (return | rank).
MODELS = {
    "lr_2":                 ("predictions_linear_regression_input=2.pkl",                 "OLS: net sentiment + log volume", "lr_2",  "return"),
    "lr_all":               ("predictions_linear_regression_input=53.pkl",                "OLS: 53 features",                "lr_all", "return"),
    "lr_text":              ("predictions_linear_regression_textonly_input=384.pkl",      "OLS: 384 text dims",              "text_ols_plain", "return"),
    "lr_text_n":            ("predictions_linear_regression_textonly_input=386.pkl",      "OLS: text + agreement",           "text_ols_raw", "return"),
    "lr_text_n_dm":         ("predictions_linear_regression_textonly_dm_input=386.pkl",   "OLS: text + agreement, de-meaned", "text_ols", "return"),
    "ridge_text_n":         ("predictions_ridge_textonly_input=386.pkl",                  "Ridge: text + agreement",         "text_ridge_raw", "return"),
    "ridge_text_n_dm":      ("predictions_ridge_textonly_dm_input=386.pkl",               "Ridge: text + agreement, de-meaned", "text_ridge", "return"),
    "lr_2_rank":            ("predictions_linear_regression_rank_input=2.pkl",            "OLS: net sentiment + log volume", "lr_2", "rank"),
    "lr_all_rank":          ("predictions_linear_regression_rank_input=53.pkl",           "OLS: 53 features",                "lr_all", "rank"),
    "lr_text_n_rank":       ("predictions_linear_regression_textonly_rank_input=386.pkl", "OLS: text + agreement",           "text_ols_raw", "rank"),
    "lr_text_n_dm_rank":    ("predictions_linear_regression_textonly_dm_rank_input=386.pkl", "OLS: text + agreement, de-meaned", "text_ols", "rank"),
    "ridge_text_n_rank":    ("predictions_ridge_textonly_rank_input=386.pkl",             "Ridge: text + agreement",         "text_ridge_raw", "rank"),
    "ridge_text_n_dm_rank": ("predictions_ridge_textonly_dm_rank_input=386.pkl",          "Ridge: text + agreement, de-meaned", "text_ridge", "rank"),
}
# The four entities drawn in the line charts (one colour each, fixed order; the others appear in the
# per-model small multiples and the summary chart only)
LINE_ENTITIES = {"lr_2": "#2a78d6", "lr_all": "#eb6834", "text_ols": "#1baf7a", "text_ridge": "#eda100"}
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"

matplotlib.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.size": 10, "axes.titlesize": 11, "legend.frameon": False, "figure.dpi": 100,
})
print(f"Figures -> {FIG_DIR}")


## 1. Load the common sample

In [ ]:
df = pd.read_pickle(MODEL_DATA_DIR / "merged_master.pkl")[["date", "permno", TARGET, "log_volume"]].copy()
for key, (fname, *_rest) in MODELS.items():
    preds = pd.read_pickle(MODEL_DATA_DIR / fname)
    df[key] = preds.set_index("index")["prediction"]
df["date"] = pd.to_datetime(df["date"])
df = df[(df["date"] >= SAMPLE_START) & (df["date"] <= SAMPLE_END) & (df["log_volume"] > 0)]
df = df.dropna(subset=[TARGET] + list(MODELS)).reset_index(drop=True)
df["y"] = df[TARGET].astype(float) - df.groupby("date")[TARGET].transform("mean").astype(float)
df["year"] = df["date"].dt.year
df["ym"] = df["date"].dt.to_period("M")
print(f"Common sample: {len(df):,} tweeted stock-days, {df['date'].nunique():,} days, {df['date'].min().date()} to {df['date'].max().date()}")


## 2. Daily rank correlations and decile returns (vectorised)

In [ ]:
# Daily Spearman = Pearson correlation of within-date percentile ranks
g = df.groupby("date")
ry = g["y"].rank(pct=True)
n_day = g["y"].transform("size")
daily_rc = {}
for key in MODELS:
    rx = g[key].rank(pct=True)
    tmp = pd.DataFrame({"date": df["date"], "x": rx, "y": ry, "xy": rx * ry, "x2": rx ** 2, "y2": ry ** 2})
    s = tmp.groupby("date").agg(n=("x", "size"), x=("x", "sum"), y=("y", "sum"), xy=("xy", "sum"), x2=("x2", "sum"), y2=("y2", "sum"))
    cov = s["xy"] / s["n"] - (s["x"] / s["n"]) * (s["y"] / s["n"])
    vx = s["x2"] / s["n"] - (s["x"] / s["n"]) ** 2
    vy = s["y2"] / s["n"] - (s["y"] / s["n"]) ** 2
    rc = cov / np.sqrt(vx * vy)
    daily_rc[key] = rc.where(s["n"] >= 10)
daily_rc = pd.DataFrame(daily_rc)

# Daily decile means of the de-meaned next-day return (decile 1 = lowest prediction)
daily_dec = {}
for key in MODELS:
    dec = np.ceil(g[key].rank(method="first", pct=True) * 10).clip(1, 10).astype(int)
    m = df.assign(dec=dec).groupby(["date", "dec"])["y"].mean().unstack()
    daily_dec[key] = m
ls = pd.DataFrame({k: v[10] - v[1] for k, v in daily_dec.items()})   # daily top-minus-bottom

def tstat(s):
    s = s.dropna(); return s.mean() / s.std() * np.sqrt(len(s))

summary = pd.DataFrame({
    "label": {k: v[1] for k, v in MODELS.items()},
    "target": {k: v[3] for k, v in MODELS.items()},
    "rank_corr": daily_rc.mean(), "rank_corr_se": daily_rc.std() / np.sqrt(daily_rc.notna().sum()),
    "rank_corr_t": daily_rc.apply(tstat),
    "spread_bp": ls.mean() * 1e4, "spread_se_bp": ls.std() / np.sqrt(ls.notna().sum()) * 1e4, "spread_t": ls.apply(tstat),
})
summary.to_csv(FIG_DIR / "model_performance_summary.csv")
print(summary.round(4).to_string())


## 3. Figures

In [ ]:
def two_panels(title, ylabel, fname, series_fn, x_is_time):
    # Two panels sharing y: return-target models on the left, rank-target on the right; one colour
    # per entity, legend plus a direct label at the right end of each line.
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
    for ax, tgt in zip(axes, ["return", "rank"]):
        ends = []
        for ent, col in LINE_ENTITIES.items():
            key = next((k for k, v in MODELS.items() if v[2] == ent and v[3] == tgt), None)
            if key is None:
                continue
            x, y = series_fn(key)
            ax.plot(x, y, color=col, linewidth=1.8, marker="o" if not x_is_time else None, markersize=5, label=MODELS[key][1])
            ends.append([x[-1], y[-1], ent.replace("text_", "text ").replace("_", " ")])
        # direct labels at the line ends, pushed apart vertically when they would collide
        lo, hi = ax.get_ylim()
        gap = 0.045 * (hi - lo)
        ends.sort(key=lambda e: e[1])
        ypos = [e[1] for e in ends]
        for i in range(1, len(ypos)):
            ypos[i] = max(ypos[i], ypos[i - 1] + gap)
        for (xe, ye, txt), yl in zip(ends, ypos):
            ax.annotate(txt, (xe, ye), xytext=(6, (yl - ye) / (hi - lo) * ax.bbox.height * 0.72),
                        textcoords="offset points", va="center", fontsize=8.5, color=INK2)
        ax.axhline(0, color=INK2, linewidth=0.8)
        ax.set_title(f"Trained on the {'next-day return' if tgt == 'return' else 'daily return rank'}")
        ax.set_ylabel(ylabel if ax is axes[0] else "")
        if not x_is_time:
            ax.set_xticks(x)
        ax.margins(x=0.12 if not x_is_time else 0.06)
    axes[0].legend(loc="upper left", fontsize=8.5)
    fig.suptitle(title, x=0.01, ha="left", fontsize=12, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(FIG_DIR / fname, dpi=200)
    plt.close(fig)
    print("saved", fname)

yearly = daily_rc.groupby(daily_rc.index.year).mean()
two_panels("Average daily rank correlation with the next-day return, by year",
           "mean daily Spearman", "rank_corr_yearly.png",
           lambda k: (yearly.index.to_numpy(), yearly[k].to_numpy()), x_is_time=False)

monthly = daily_rc.groupby(daily_rc.index.to_period("M")).mean()
rolling = monthly.rolling(12, min_periods=6).mean()
two_panels("Average daily rank correlation, 12-month rolling mean of monthly averages",
           "mean daily Spearman (12m rolling)", "rank_corr_monthly_rolling12.png",
           lambda k: (rolling.index.to_timestamp().to_numpy(), rolling[k].to_numpy()), x_is_time=True)

cum = ls.fillna(0).cumsum() * 100
two_panels("Cumulative top-minus-bottom decile return (equal-weighted, de-meaned by date)",
           "cumulative return, %", "cumulative_decile_spread.png",
           lambda k: (cum.index.to_numpy(), cum[k].to_numpy()), x_is_time=True)


In [ ]:
# Decile profiles: one small panel per model, single hue, zero baseline, D1/D10 labelled
keys = list(MODELS)
ncol = 4
nrow = int(np.ceil(len(keys) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(14, 3.1 * nrow), sharey=True)
prof = pd.DataFrame({k: v.mean() * 1e4 for k, v in daily_dec.items()})
for ax, key in zip(axes.flat, keys):
    vals = prof[key]
    ax.bar(vals.index, vals.values, color="#2a78d6", width=0.72)
    ax.axhline(0, color=INK2, linewidth=0.8)
    ax.set_title(f"{MODELS[key][1]}\n({MODELS[key][3]} target)", fontsize=9)
    ax.set_xticks(range(1, 11)); ax.tick_params(labelsize=8)
    for d in (1, 10):
        ax.annotate(f"{vals[d]:.0f}", (d, vals[d]), xytext=(0, 4 if vals[d] >= 0 else -10),
                    textcoords="offset points", ha="center", fontsize=8, color=INK2)
    ax.grid(axis="x", visible=False)
for ax in axes.flat[len(keys):]:
    ax.set_visible(False)
for ax in axes[:, 0]:
    ax.set_ylabel("mean next-day return, bp")
for ax in axes[-1, :]:
    ax.set_xlabel("daily prediction decile")
fig.suptitle("Mean de-meaned next-day return by daily prediction decile (common sample, 2012-2022)",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(FIG_DIR / "decile_profiles.png", dpi=200)
plt.close(fig)
print("saved decile_profiles.png")


In [ ]:
# Summary: rank correlation and decile spread with 95% bands, every model, sorted by rank correlation
order = summary.sort_values("rank_corr").index
labels = [f"{summary.loc[k, 'label']}  [{summary.loc[k, 'target']}]" for k in order]
ypos = np.arange(len(order))
fig, axes = plt.subplots(1, 2, figsize=(14, 6.2), sharey=True)
for ax, col, se, xlab, fmt in [(axes[0], "rank_corr", "rank_corr_se", "mean daily rank correlation", "{:.4f}"),
                               (axes[1], "spread_bp", "spread_se_bp", "top-minus-bottom decile, bp/day", "{:.1f}")]:
    v, e = summary.loc[order, col].to_numpy(), 1.96 * summary.loc[order, se].to_numpy()
    ax.errorbar(v, ypos, xerr=e, fmt="o", color="#2a78d6", ecolor="#9ec5f4", elinewidth=2, markersize=7, capsize=0)
    ax.axvline(0, color=INK2, linewidth=0.8)
    for yy, val, err in zip(ypos, v, e):
        ax.annotate(fmt.format(val), (val + err, yy), xytext=(5, 0), textcoords="offset points", va="center", fontsize=8.5, color=INK2)
    ax.set_xlabel(xlab); ax.grid(axis="y", visible=False); ax.margins(x=0.18)
axes[0].set_yticks(ypos); axes[0].set_yticklabels(labels, fontsize=9)
fig.suptitle("Full-sample performance with 95% confidence bands (common sample, 2012-2022)",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIG_DIR / "summary_rank_corr_and_spread.png", dpi=200)
plt.close(fig)
print("saved summary_rank_corr_and_spread.png")
print("\nFiles:", sorted(p.name for p in FIG_DIR.iterdir()))
